# CMR Report on Colab — No data upload needed

This notebook runs the **PTB-XL → CMR report** pipeline on Colab with GPU.

**You don’t need to download or upload any data.** It uses the small **test PTB data** that’s already in the repo (`data/ptbxl_test/`). The notebook converts that into the format the pipeline expects and runs it.

---

### Step 0: Open in Colab and turn on GPU

1. Click: **[Open in Colab](https://colab.research.google.com/github/JeremieTarantop/Cardiac-Diagnostic-CMR-report-/blob/main/notebooks/CMR_report_Colab_TestData.ipynb)**  
2. **Runtime → Change runtime type → T4 GPU** (or better) → Save.
3. Run the cells **in order** (Runtime → Run all, or run each cell with Shift+Enter).

## 1. Clone the repo

This brings in the code and the test data (`data/ptbxl_test/`).

In [ ]:
!git clone https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-.git repo_cmr
%cd repo_cmr

## 2. Install dependencies

Installs PyTorch, Transformers, and WFDB (to read the test ECG files).

In [ ]:
!pip install -q transformers torch pandas numpy scipy wfdb

## 3. Prepare test data (no upload needed)

Converts the WFDB files in `data/ptbxl_test/` into the format the pipeline expects and creates minimal metadata. Everything stays inside the repo.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import wfdb
except ImportError:
    !pip install -q wfdb
    import wfdb

REPO = Path(".")
TEST_DIR = REPO / "data" / "ptbxl_test" / "00000"
OUT_NPY_DIR = REPO / "data" / "ptbxl_pclr_format" / "00000"
OUT_NPY_DIR.mkdir(parents=True, exist_ok=True)
(REPO / "data" / "ptbxl_with_labels").mkdir(parents=True, exist_ok=True)

TARGET_SAMPLES = 4096
N_LEADS = 12

hea_files = sorted(TEST_DIR.glob("*.hea"))
if not hea_files:
    raise FileNotFoundError(f"No .hea files in {TEST_DIR}. Is data/ptbxl_test in the repo?")

print(f"Found {len(hea_files)} test records in data/ptbxl_test/00000/")

labels_rows = []
db_rows = []

for i, hea_path in enumerate(hea_files[:50], start=1):  # up to 50
    rec_name = hea_path.stem
    rec_path = str(hea_path.with_suffix(""))
    rec = wfdb.rdrecord(rec_path)
    sig = rec.p_signal  # (n_samples, n_sig)
    if sig.shape[1] != N_LEADS:
        continue
    if sig.shape[0] >= TARGET_SAMPLES:
        sig = sig[:TARGET_SAMPLES, :]
    else:
        pad = np.zeros((TARGET_SAMPLES - sig.shape[0], N_LEADS), dtype=sig.dtype)
        sig = np.vstack([sig, pad])
    npy_path = OUT_NPY_DIR / f"{rec_name}.npy"
    np.save(npy_path, sig)
    rel_path = npy_path.relative_to(REPO)
    labels_rows.append({
        "ecg_id": i,
        "ecg_file": str(rel_path),
        "age": 50.0,
        "sex": 1,
        "height": "", "weight": "", "scp_codes": "{}", "heart_axis": "",
        "infarction_stadium1": "", "infarction_stadium2": "", "baseline_drift": "",
        "static_noise": "", "burst_noise": "", "electrodes_problems": "", "extra_beats": "", "pacemaker": "",
    })
    db_rows.append({
        "ecg_id": i,
        "patient_id": 0,
        "age": 50.0,
        "sex": 1,
        "height": "", "weight": "", "nurse": "", "site": "", "device": "", "recording_date": "",
        "report": f"Test record {i} from PTB-XL (WFDB)",
        "scp_codes": "{}",
        "heart_axis": "",
        "infarction_stadium1": "", "infarction_stadium2": "", "validated_by": "", "second_opinion": "",
        "initial_autogenerated_report": "", "validated_by_human": "",
        "baseline_drift": "", "static_noise": "", "burst_noise": "", "electrodes_problems": "", "extra_beats": "", "pacemaker": "",
        "strat_fold": "", "filename_lr": "", "filename_hr": "",
    })

pd.DataFrame(labels_rows).to_csv(REPO / "data" / "ptbxl_with_labels" / "ecg_with_labels.csv", index=False)
pd.DataFrame(db_rows).to_csv(REPO / "data" / "ptbxl_database.csv", index=False)

print(f"Created {len(labels_rows)} records in data/ptbxl_pclr_format and metadata CSVs.")
print("You can run the CMR pipeline with --ecg-id 1 to", len(labels_rows))
print("Done. Run the next cell.")

## 4. Generate the CMR report — TinyLlama (GPU)

Runs TinyLlama on the first test ECG and prints the CMR report. Uses GPU if available.

In [ ]:
import os
os.environ["USE_TRANSFORMERS"] = "1"
os.environ["USE_CUDA"] = "1"

!python -m ecg_to_cmr_report.e_to_c_llama1 --ecg-id 1

## 4b. (Optional) Generate with MedGemma 4B (medical LLM, GPU)

Uses the same test data. **Requires a Hugging Face token:** accept [MedGemma terms](https://huggingface.co/google/medgemma-4b-it), then set your token in Colab (e.g. **Secrets** or run the line below with your token). Slower than TinyLlama but more medical wording.

In [ ]:
# Optional: set your HF token (or add it in Colab Secrets as HF_TOKEN)
# os.environ["HF_TOKEN"] = "your_hf_token_here"
!python -m ecg_to_cmr_report.e_to_c_medgemma4b --ecg-id 1 --max-tokens 128

## 5. (Optional) Try another test record

Change `--ecg-id` to 2, 3, … up to the number of test records prepared above.

In [ ]:
# !python -m ecg_to_cmr_report.e_to_c_llama1 --ecg-id 2